# 02 · Metric, seasonal-naive baseline & features (M2)

Project FORESIGHT · NorthBay Living · Zidio Development Data Science internship · Eswar Mahalingam

In [1]:
import sys, json, pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
pd.set_option('display.width', 160); pd.set_option('display.max_columns', 30)
from IPython.display import Image, display

## 1. Frame the metric and horizon
**WAPE** = Σ|actual − forecast| ÷ Σ actual, over an **8-week** horizon at SKU-week grain. Bias = Σ(forecast − actual) ÷ Σ actual as the secondary check.

In [2]:
from forecast import load_weekly, make_rows, wape, bias, FEATURES, HORIZON
wk = load_weekly(); print(HORIZON, 'week horizon;', wk.week_end.nunique(), 'weeks;', wk.sku_id.nunique(), 'SKUs')

8 week horizon; 104 weeks; 50 SKUs


## 2. Seasonal-naive baseline
Forecast for week *t+h* = actual units in week *t+h−52*. This is the bar every model must clear.

In [3]:
origin = pd.Timestamp('2025-08-10')
rows = make_rows(wk, origin).dropna(subset=['y'])
print('origin', origin.date(), '| seasonal-naive WAPE', round(wape(rows.y, rows.naive),3), '| bias', round(bias(rows.y, rows.naive),3))
rows.groupby('h').apply(lambda t: round(wape(t.y, t.naive),3)).rename('naive WAPE by horizon')

origin 2025-08-10 | seasonal-naive WAPE 0.121 | bias 0.001


h
1    0.122
2    0.123
3    0.112
4    0.134
5    0.121
6    0.108
7    0.121
8    0.131
Name: naive WAPE by horizon, dtype: float64

## 3. Engineered features (all computed from history ≤ origin — no leakage)
Lags 1–13, rolling means/std, trend, last-year same-week (lag 52) and its 3-week average, this-year-vs-last-year level ratio, planned promo/holiday days of the *target* week, week-of-year, month, season, weeks since launch, price, SKU and category identity, and the horizon *h* itself (direct multi-horizon).

In [4]:
print(FEATURES)
rows[['sku_id','week_end','h','lag1','mean4','lag52_target','promo_days_target','naive','y']].head(10)

['h', 'lag1', 'lag2', 'lag3', 'lag4', 'lag8', 'lag13', 'mean4', 'mean8', 'mean13', 'std4', 'trend4_13', 'lag52_target', 'lag52_mean3', 'level_ratio_52', 'promo_days_target', 'holiday_days_target', 'week_of_year', 'month', 'weeks_since_launch', 'unit_price', 'sku_id', 'category', 'season']


,sku_id,week_end,h,lag1,mean4,lag52_target,promo_days_target,naive,y
0,SKU001,2025-08-17,1,77,101.75,89,0,89,90
1,SKU002,2025-08-17,1,60,70.00,58,0,58,70
2,SKU003,2025-08-17,1,38,35.50,37,0,37,39
3,SKU004,2025-08-17,1,26,30.50,30,0,30,26
4,SKU005,2025-08-17,1,96,110.25,103,0,103,114
5,SKU006,2025-08-17,1,46,45.75,39,0,39,38
6,SKU007,2025-08-17,1,145,163.50,164,0,164,158
7,SKU008,2025-08-17,1,101,107.00,99,0,99,126
8,SKU009,2025-08-17,1,66,68.75,70,0,70,61
9,SKU010,2025-08-17,1,95,103.75,103,0,103,113


## 4. Leakage check
For any origin, `lag1` must equal the last observed week ≤ origin and all target weeks must be strictly after the origin.

In [5]:
hist = wk[wk.week_end <= origin].pivot(index='week_end', columns='sku_id', values='units')
assert np.allclose(rows[rows.h==1].set_index('sku_id').lag1.reindex(hist.columns), hist.iloc[-1])
assert (rows.week_end > origin).all(); print('leakage checks passed')

leakage checks passed
